In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rng = np.random.default_rng(9)
n = 200
df = pd.DataFrame({
    "TV": rng.uniform(0, 300, n),
    "Radio": rng.uniform(0, 50, n),
    "Newspaper": rng.uniform(0, 100, n),
})
df["Sales"] = 3 + 0.045 * df["TV"] + 0.19 * df["Radio"] + 0.002 * df["Newspaper"] + rng.normal(0, 1.2, n)
df.to_csv("advertising.csv", index=False)

df = pd.read_csv("advertising.csv")
print(df.head(), "\n")
print("Correlation with Sales:\n", df.corr()["Sales"].round(3), "\n")

X = df[["TV", "Radio", "Newspaper"]]
y = df["Sales"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LinearRegression().fit(X_train, y_train)
rf = RandomForestRegressor(n_estimators=200, random_state=42).fit(X_train, y_train)

for name, pred in [("Linear Regression", lr.predict(X_test)),
                   ("Random Forest", rf.predict(X_test))]:
    print(f"{name}: R2 = {r2_score(y_test, pred):.4f}, "
          f"MAE = {mean_absolute_error(y_test, pred):.4f}, "
          f"RMSE = {np.sqrt(mean_squared_error(y_test, pred)):.4f}")

print("\nLinear Regression coefficients:", dict(zip(X.columns, lr.coef_.round(4))))

future = pd.DataFrame({"TV": [150, 220, 280], "Radio": [20, 30, 45], "Newspaper": [30, 40, 20]})
future["PredictedSales"] = lr.predict(future).round(2)
print("\nFuture sales predictions:\n", future)

plt.scatter(y_test, lr.predict(X_test), s=15)
plt.plot([y.min(), y.max()], [y.min(), y.max()], "r--")
plt.xlabel("Actual sales")
plt.ylabel("Predicted sales")
plt.title("Future Sales Prediction (Linear Regression)")
plt.show()